In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.evaluate_models import eval_clf

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("Session 32 imports successful")

Session 32 imports successful


In [2]:
df = pd.read_csv(
    "../data/student-mat.csv",
    sep=";"
)

df_encoded = pd.get_dummies(
    df,
    drop_first=True
)

print("Dataset loaded.")
print("Original shape:", df.shape)
print("Encoded shape:", df_encoded.shape)

Dataset loaded.
Original shape: (395, 33)
Encoded shape: (395, 42)


In [3]:
df_encoded["at_risk"] = (
    df_encoded["G3"] < 10
).astype(int)

print("Target created.")

print("\nClass counts:")
print(
    df_encoded["at_risk"]
    .value_counts()
    .sort_index()
)

print("\nClass proportions:")
print(
    df_encoded["at_risk"]
    .value_counts(normalize=True)
    .sort_index()
    .round(3)
)

Target created.

Class counts:
at_risk
0    265
1    130
Name: count, dtype: int64

Class proportions:
at_risk
0    0.671
1    0.329
Name: proportion, dtype: float64


In [4]:
X = df_encoded.drop(
    columns=["G3", "at_risk"]
).copy()

y = df_encoded["at_risk"].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (395, 41)
Target shape: (395,)


In [5]:
Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train/Test split complete.")

print("Xtr_c:", Xtr_c.shape)
print("Xte_c:", Xte_c.shape)

print("ytr_c:", ytr_c.shape)
print("yte_c:", yte_c.shape)

Train/Test split complete.
Xtr_c: (316, 41)
Xte_c: (79, 41)
ytr_c: (316,)
yte_c: (79,)


In [6]:
logistic_pipeline = Pipeline([
    ("standardscaler", StandardScaler()),
    (
        "logisticregression",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

print("Logistic Regression pipeline created.")

Logistic Regression pipeline created.


In [7]:
for step_name, step_object in logistic_pipeline.steps:
    print(step_name, "->", step_object)

standardscaler -> StandardScaler()
logisticregression -> LogisticRegression(max_iter=1000, random_state=42)


In [8]:
logistic_pipeline.fit(
    Xtr_c,
    ytr_c
)

print("Logistic Regression training complete.")

Logistic Regression training complete.


In [9]:
print(
    logistic_pipeline.named_steps[
        "logisticregression"
    ]
)

LogisticRegression(max_iter=1000, random_state=42)


In [10]:
y_pred_logistic = logistic_pipeline.predict(
    Xte_c
)

y_proba_logistic = (
    logistic_pipeline
    .predict_proba(Xte_c)[:, 1]
)

print(
    "Predicted labels:",
    len(y_pred_logistic)
)

print(
    "Predicted probabilities:",
    len(y_proba_logistic)
)

Predicted labels: 79
Predicted probabilities: 79


In [11]:
print(
    "First 10 class predictions:"
)
print(
    y_pred_logistic[:10]
)

print(
    "\nFirst 10 probabilities:"
)
print(
    y_proba_logistic[:10]
)

First 10 class predictions:
[1 0 1 0 1 0 0 0 1 0]

First 10 probabilities:
[9.97811518e-01 4.33214763e-01 8.78430931e-01 3.03186854e-01
 9.86612813e-01 4.51457528e-03 1.46463242e-04 8.99027039e-06
 9.32195557e-01 1.08767449e-03]


In [12]:
logistic_metrics = eval_clf(
    yte_c,
    y_pred_logistic,
    y_proba_logistic
)

print("Logistic Regression Results")
print(logistic_metrics)

Logistic Regression Results
{'accuracy': 0.9240506329113924, 'precision': 0.9545454545454546, 'recall': 0.8076923076923077, 'f1': 0.875, 'roc_auc': 0.9709724238026125}


In [13]:
required_metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]

for metric_name in required_metrics:
    print(
        metric_name,
        ":",
        logistic_metrics[metric_name]
    )

accuracy : 0.9240506329113924
precision : 0.9545454545454546
recall : 0.8076923076923077
f1 : 0.875
roc_auc : 0.9709724238026125


In [14]:
logistic_row = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": logistic_metrics["accuracy"],
        "Precision": logistic_metrics["precision"],
        "Recall": logistic_metrics["recall"],
        "F1": logistic_metrics["f1"],
        "ROC_AUC": logistic_metrics["roc_auc"]
    }
])

logistic_row.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.9241,0.9545,0.8077,0.875,0.971


In [15]:
print("Test observations:", len(yte_c))
print("Predictions:", len(y_pred_logistic))
print("Probabilities:", len(y_proba_logistic))

assert len(yte_c) == len(y_pred_logistic)
assert len(yte_c) == len(y_proba_logistic)

print("Prediction counts validated.")

Test observations: 79
Predictions: 79
Probabilities: 79
Prediction counts validated.


In [16]:
y_true_array = np.asarray(
    yte_c
).astype(int)

y_pred_array = np.asarray(
    y_pred_logistic
).astype(int)

true_negative = int(
    (
        (y_true_array == 0)
        & (y_pred_array == 0)
    ).sum()
)

false_positive = int(
    (
        (y_true_array == 0)
        & (y_pred_array == 1)
    ).sum()
)

false_negative = int(
    (
        (y_true_array == 1)
        & (y_pred_array == 0)
    ).sum()
)

true_positive = int(
    (
        (y_true_array == 1)
        & (y_pred_array == 1)
    ).sum()
)

print("TN:", true_negative)
print("FP:", false_positive)
print("FN:", false_negative)
print("TP:", true_positive)

TN: 52
FP: 1
FN: 5
TP: 21


In [17]:
confusion_summary = pd.DataFrame([
    {
        "True Negative": true_negative,
        "False Positive": false_positive,
        "False Negative": false_negative,
        "True Positive": true_positive
    }
])

confusion_summary

,True Negative,False Positive,False Negative,True Positive
0,52,1,5,21


In [18]:
false_positive_rate = (
    false_positive
    / (false_positive + true_negative)
)

false_negative_rate = (
    false_negative
    / (false_negative + true_positive)
)

print(
    f"False Positive Rate: {false_positive_rate:.4f}"
)

print(
    f"False Negative Rate: {false_negative_rate:.4f}"
)

False Positive Rate: 0.0189
False Negative Rate: 0.1923


In [19]:
session32_summary = f"""
SESSION 32 LOGISTIC REGRESSION SUMMARY

Accuracy: {logistic_metrics['accuracy']:.4f}
Precision: {logistic_metrics['precision']:.4f}
Recall: {logistic_metrics['recall']:.4f}
F1 Score: {logistic_metrics['f1']:.4f}
ROC-AUC: {logistic_metrics['roc_auc']:.4f}

Confusion Matrix

TN = {true_negative}
FP = {false_positive}
FN = {false_negative}
TP = {true_positive}
""".strip()

print(session32_summary)

SESSION 32 LOGISTIC REGRESSION SUMMARY

Accuracy: 0.9241
Precision: 0.9545
Recall: 0.8077
F1 Score: 0.8750
ROC-AUC: 0.9710

Confusion Matrix

TN = 52
FP = 1
FN = 5
TP = 21
